# LeetCode 2241 — Design an ATM Machine bằng Python

Link đề: https://leetcode.com/problems/design-an-atm-machine/ · Độ khó: **Medium** · Chủ đề: Greedy, Design, Array

Skeleton LeetCode:

```python
class ATM:
    def __init__(self):
        ...
    def deposit(self, banknotesCount: List[int]) -> None:
        ...
    def withdraw(self, amount: int) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Đọc kỹ **luật nghiệp vụ**: máy rút tiền **ưu tiên mệnh giá lớn** và **không được** chọn cách khác dù cách khác khả thi — đây không phải bài "đổi tiền tối ưu".
2. Cài đặt tham lam với **chia nguyên + `min`** thay vì lấy từng tờ (vì số tờ tới `10^9`).
3. Đảm bảo **tính nguyên tử (atomicity)**: rút thất bại thì trạng thái máy **không đổi** — tính kế hoạch trên bản nháp, chỉ "commit" khi thành công.
4. Thiết kế lớp có trạng thái: kiểm thử bằng **chuỗi lệnh ngẫu nhiên** so với một bản mô phỏng ngây thơ.
5. Nhận ra các lỗi thiết kế kinh điển: sửa trạng thái trước khi biết thành công, trả về tham chiếu tới trạng thái nội bộ, nhầm thứ tự mệnh giá.

## 2. Đề bài & ví dụ

Máy ATM chứa tờ tiền 5 mệnh giá `20, 50, 100, 200, 500`, ban đầu rỗng.

- `deposit(banknotesCount)`: nạp thêm tờ, mảng 5 phần tử theo thứ tự `$20, $50, $100, $200, $500`.
- `withdraw(amount)`: máy **ưu tiên tờ lớn**: đi từ `$500` xuống `$20`, mỗi mệnh giá lấy **nhiều nhất có thể** (không vượt số còn phải trả và số tờ đang có). Nếu cuối cùng trả đủ, trả về mảng 5 số tờ đã đưa (thứ tự `$20 … $500`) và trừ khỏi máy; nếu không, trả về `[-1]` và **không** rút tờ nào.

Lưu ý quan trọng của đề: rút `$600` khi máy có `3×$200` và `1×$500` bị **từ chối** — máy lấy `$500` trước rồi không trả nổi `$100` còn lại; máy **không** được dùng `3×$200` thay thế.

**Ví dụ 1:**
```
ATM(); deposit([0,0,1,2,1]); withdraw(600) -> [0,0,1,0,1]
deposit([0,1,0,1,1]); withdraw(600) -> [-1]; withdraw(550) -> [0,1,0,0,1]
```

**Ràng buộc:** `0 ≤ banknotesCount[i] ≤ 10^9`, `1 ≤ amount ≤ 10^9`, tối đa `5000` lời gọi; tổng số tờ nạp `≤ 10^9`.

In [1]:
import random
import time
from itertools import product
from typing import List

DENOMS = [20, 50, 100, 200, 500]
example_ops = ["ATM", "deposit", "withdraw", "deposit", "withdraw", "withdraw"]
example_args = [[], [[0, 0, 1, 2, 1]], [600], [[0, 1, 0, 1, 1]], [600], [550]]
example_out = [None, None, [0, 0, 1, 0, 1], None, [-1], [0, 1, 0, 0, 1]]
for op, arg, out in zip(example_ops, example_args, example_out):
    print(f"{op}({arg}) -> {out}")

ATM([]) -> None
deposit([[0, 0, 1, 2, 1]]) -> None
withdraw([600]) -> [0, 0, 1, 0, 1]
deposit([[0, 1, 0, 1, 1]]) -> None
withdraw([600]) -> [-1]
withdraw([550]) -> [0, 1, 0, 0, 1]


## 3. Kiến thức nền

### 3.1 Chia nguyên + `min`

Ở mệnh giá `d`, còn phải trả `rem`, máy đang có `c` tờ: số tờ lấy là `take = min(c, rem // d)`. Sau đó `rem -= take · d`. Một phép tính thay cho vòng lặp "lấy từng tờ".

In [2]:
rem, d, c = 1730, 500, 2
take = min(c, rem // d)
print(f"còn {rem}, mệnh giá {d}, có {c} tờ -> lấy {take} tờ, còn phải trả {rem - take * d}")

còn 1730, mệnh giá 500, có 2 tờ -> lấy 2 tờ, còn phải trả 730


### 3.2 Thao tác nguyên tử: tính trên bản nháp, commit khi thành công

Mẫu chung khi một thao tác có thể thất bại giữa chừng: (1) tính kế hoạch `plan` mà **không** sửa trạng thái; (2) nếu thất bại, trả lỗi; (3) nếu thành công, áp dụng `plan` vào trạng thái. Giống giao dịch (transaction) trong cơ sở dữ liệu.

In [3]:
state = [5, 5]
def try_take(state, need):
    plan = [min(s, n) for s, n in zip(state, need)]      # chỉ tính, chưa sửa
    if plan != need:
        return None                                      # thất bại: state nguyên vẹn
    for i, p in enumerate(plan):
        state[i] -= p                                    # commit
    return plan

print(try_take(state, [2, 7]), state)
print(try_take(state, [2, 3]), state)

None [5, 5]
[2, 3] [3, 2]


### 3.3 Danh sách trả về phải là **bản sao**

Nếu `withdraw` trả về chính list nội bộ (hoặc `deposit` lưu chính list người gọi truyền vào), người gọi sửa list đó sẽ làm hỏng trạng thái máy. Python truyền **tham chiếu** cho list.

In [4]:
inner = [1, 2, 3]
alias = inner          # cùng một đối tượng
alias[0] = 99
print("inner sau khi sửa alias:", inner, "| alias is inner:", alias is inner)

inner sau khi sửa alias: [99, 2, 3] | alias is inner: True


## 4. Ý tưởng nền tảng / trực giác

Máy hành xử giống một nhân viên thu ngân "máy móc": luôn đưa tờ to nhất còn đưa được. Ta chỉ cần **mô phỏng đúng** quy tắc đó — không tối ưu gì cả.

Trace bằng tay ví dụ 1, lần rút thứ hai (`withdraw(600)`, máy `[0, 1, 0, 3, 1]`):

| Mệnh giá | Có | Còn phải trả | Lấy | Còn lại |
|---|---|---|---|---|
| 500 | 1 | 600 | `min(1, 600//500 = 1) = 1` | 100 |
| 200 | 3 | 100 | `min(3, 100//200 = 0) = 0` | 100 |
| 100 | 0 | 100 | 0 | 100 |
| 50 | 1 | 100 | `min(1, 2) = 1` | 50 |
| 20 | 0 | 50 | 0 | **50 ≠ 0** ⇒ `[-1]` |

Dù `3 × 200 = 600` khả thi, máy vẫn từ chối — đúng như đề yêu cầu.

### 4.1 Vì sao đây **không** phải bài "đổi tiền"?

Bài đổi tiền (Coin Change) hỏi "có cách nào không?" và tham lam có thể sai với hệ mệnh giá này (vd. `60 = 3×20` nhưng tham lam lấy `50` trước rồi kẹt `10`). Ở đây **đề định nghĩa** hành vi là tham lam, nên "sai" của tham lam lại chính là **đáp án đúng** theo đặc tả. Viết DP "thông minh hơn" là **sai đề**.

## 5. Cách 1 — Brute Force / baseline: mô phỏng từng tờ

**Ý tưởng:** lặp từ mệnh giá lớn xuống nhỏ, **từng tờ một**: còn tờ và mệnh giá ≤ số còn phải trả thì lấy một tờ. Đây là cách diễn đạt sát nghĩa nhất của đặc tả — dùng làm **chuẩn đối chiếu** (oracle). Chỉ dùng khi số tờ nhỏ.

In [5]:
class ATMBrute:
    def __init__(self):
        self.cnt = [0] * 5
    def deposit(self, banknotesCount):
        for i in range(5):
            self.cnt[i] += banknotesCount[i]
    def withdraw(self, amount):
        draft = self.cnt[:]                     # bản nháp
        give = [0] * 5; rem = amount
        for i in range(4, -1, -1):
            while draft[i] > 0 and DENOMS[i] <= rem:      # lấy từng tờ
                draft[i] -= 1; give[i] += 1; rem -= DENOMS[i]
        if rem != 0:
            return [-1]
        self.cnt = draft
        return give

def run(cls, ops, args):
    obj, out = None, []
    for op, arg in zip(ops, args):
        if op == "ATM":
            obj = cls(); out.append(None)
        else:
            out.append(getattr(obj, op)(*arg))
    return out

got = run(ATMBrute, example_ops, example_args)
print(got); assert got == example_out

[None, None, [0, 0, 1, 0, 1], None, [-1], [0, 1, 0, 0, 1]]


**Độ phức tạp:** mỗi `withdraw` tốn `O(số tờ được đưa)` — có thể tới `amount / 20 = 5·10^7` vòng ⇒ quá chậm với `10^9`. Space `O(1)`.

**Nút thắt:** vòng `while` lấy từng tờ chỉ để tính `min(cnt, rem // d)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** vòng "lấy từng tờ" ở mỗi mệnh giá dừng khi hết tờ **hoặc** khi `d > rem`. Số tờ lấy được là `min(cnt[i], rem // d)` — tính bằng một phép chia. Mỗi `withdraw` thành `O(5)`.

**Insight 2 (nguyên tử):** tính `take[i]` cho cả 5 mệnh giá mà **không** sửa `cnt`; chỉ khi `rem == 0` mới trừ `cnt[i] -= take[i]`.

**Quy tắc "nếu… thì…":**
- **Nếu** đề mô tả **hành vi cụ thể** (ưu tiên lớn, không được thay thế) **thì** mô phỏng đúng hành vi, đừng "tối ưu" thay đề.
- **Nếu** một thao tác có thể thất bại **thì** tính trước, commit sau.
- **Nếu** số lượng lớn (`10^9`) **thì** thay vòng lặp từng đơn vị bằng chia nguyên.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam bằng chia nguyên + commit (bản nộp chính)

In [6]:
class ATMGreedy:
    def __init__(self):
        self.cnt = [0] * 5
    def deposit(self, banknotesCount):
        for i, x in enumerate(banknotesCount):
            self.cnt[i] += x
    def withdraw(self, amount):
        take = [0] * 5
        for i in range(4, -1, -1):
            take[i] = min(self.cnt[i], amount // DENOMS[i])
            amount -= take[i] * DENOMS[i]
        if amount:
            return [-1]
        for i in range(5):
            self.cnt[i] -= take[i]
        return take

got = run(ATMGreedy, example_ops, example_args)
print(got); assert got == example_out

[None, None, [0, 0, 1, 0, 1], None, [-1], [0, 1, 0, 0, 1]]


**Độ phức tạp Cách 2:** `deposit` và `withdraw` đều `O(5) = O(1)`. Space `O(1)`.

### 7.2 Cách 3 — Lưu tổng tiền để từ chối sớm

Giữ thêm `self.total` (tổng tiền trong máy). Nếu `amount > total` thì chắc chắn thất bại ⇒ trả `[-1]` ngay. Không đổi độ phức tạp, nhưng minh họa **bất biến phụ** cần cập nhật đồng bộ ở cả `deposit` và `withdraw`.

In [7]:
class ATMWithTotal:
    def __init__(self):
        self.cnt = [0] * 5; self.total = 0
    def deposit(self, banknotesCount):
        for i, x in enumerate(banknotesCount):
            self.cnt[i] += x; self.total += x * DENOMS[i]
    def withdraw(self, amount):
        if amount > self.total:                 # từ chối sớm
            return [-1]
        take = [0] * 5; rem = amount
        for i in range(4, -1, -1):
            take[i] = min(self.cnt[i], rem // DENOMS[i]); rem -= take[i] * DENOMS[i]
        if rem:
            return [-1]
        for i in range(5):
            self.cnt[i] -= take[i]
        self.total -= amount                     # giữ bất biến total = Σ cnt·d
        return take

got = run(ATMWithTotal, example_ops, example_args)
print(got); assert got == example_out

[None, None, [0, 0, 1, 0, 1], None, [-1], [0, 1, 0, 0, 1]]


**Độ phức tạp Cách 3:** `O(1)` mỗi thao tác; space `O(1)`.

### 7.3 Cách 4 — Tổng quát hóa: mệnh giá bất kỳ, dùng `dict`

Cho máy nhận danh sách mệnh giá tùy ý (sắp xếp giảm dần nội bộ). Tách hàm thuần `plan_withdraw(cnt, amount)` (không có trạng thái) — dễ kiểm thử riêng.

In [8]:
def plan_withdraw(denoms_desc, cnt, amount):
    """Hàm thuần: trả về dict {mệnh giá: số tờ} hoặc None nếu không trả đủ."""
    take = {}
    for d in denoms_desc:
        take[d] = min(cnt[d], amount // d); amount -= take[d] * d
    return take if amount == 0 else None

class ATMGeneral:
    def __init__(self, denoms=(20, 50, 100, 200, 500)):
        self.denoms = list(denoms)                       # thứ tự để nhận/trả mảng
        self.desc = sorted(self.denoms, reverse=True)
        self.cnt = {d: 0 for d in self.denoms}
    def deposit(self, banknotesCount):
        for d, x in zip(self.denoms, banknotesCount):
            self.cnt[d] += x
    def withdraw(self, amount):
        take = plan_withdraw(self.desc, self.cnt, amount)
        if take is None:
            return [-1]
        for d, t in take.items():
            self.cnt[d] -= t
        return [take[d] for d in self.denoms]

got = run(ATMGeneral, example_ops, example_args)
print(got); assert got == example_out

[None, None, [0, 0, 1, 0, 1], None, [-1], [0, 1, 0, 0, 1]]


**Độ phức tạp Cách 4:** `O(D)` mỗi thao tác với `D` mệnh giá (`O(D log D)` một lần khi khởi tạo). Space `O(D)`.

## 8. TRACE từng bước

Trace Cách 2 trên toàn bộ ví dụ 1: in trạng thái máy sau mỗi lệnh và chi tiết từng mệnh giá trong `withdraw`.

| Lệnh | Trạng thái trước | Kết quả | Trạng thái sau |
|---|---|---|---|
| deposit([0,0,1,2,1]) | [0,0,0,0,0] | — | [0,0,1,2,1] |
| withdraw(600) | [0,0,1,2,1] | [0,0,1,0,1] | [0,0,0,2,0] |
| deposit([0,1,0,1,1]) | [0,0,0,2,0] | — | [0,1,0,3,1] |
| withdraw(600) | [0,1,0,3,1] | [-1] | [0,1,0,3,1] (không đổi) |
| withdraw(550) | [0,1,0,3,1] | [0,1,0,0,1] | [0,0,0,3,0] |

In [9]:
class ATMTrace(ATMGreedy):
    def withdraw(self, amount):
        print(f"  withdraw({amount}) với máy {self.cnt}")
        take = [0] * 5; rem = amount
        for i in range(4, -1, -1):
            take[i] = min(self.cnt[i], rem // DENOMS[i])
            print(f"    ${DENOMS[i]:3d}: có {self.cnt[i]}, cần ≤ {rem // DENOMS[i]} -> lấy {take[i]}, còn {rem - take[i] * DENOMS[i]}")
            rem -= take[i] * DENOMS[i]
        if rem:
            print("    -> còn dư", rem, "=> [-1], máy giữ nguyên"); return [-1]
        for i in range(5): self.cnt[i] -= take[i]
        print("    -> thành công", take, "| máy còn", self.cnt); return take

atm = ATMTrace()
for op, arg in zip(example_ops[1:], example_args[1:]):
    res = getattr(atm, op)(*arg)
    print(f"{op}{tuple(arg)} -> {res} | máy: {atm.cnt}")

deposit([0, 0, 1, 2, 1],) -> None | máy: [0, 0, 1, 2, 1]
  withdraw(600) với máy [0, 0, 1, 2, 1]
    $500: có 1, cần ≤ 1 -> lấy 1, còn 100
    $200: có 2, cần ≤ 0 -> lấy 0, còn 100
    $100: có 1, cần ≤ 1 -> lấy 1, còn 0
    $ 50: có 0, cần ≤ 0 -> lấy 0, còn 0
    $ 20: có 0, cần ≤ 0 -> lấy 0, còn 0
    -> thành công [0, 0, 1, 0, 1] | máy còn [0, 0, 0, 2, 0]
withdraw(600,) -> [0, 0, 1, 0, 1] | máy: [0, 0, 0, 2, 0]
deposit([0, 1, 0, 1, 1],) -> None | máy: [0, 1, 0, 3, 1]
  withdraw(600) với máy [0, 1, 0, 3, 1]
    $500: có 1, cần ≤ 1 -> lấy 1, còn 100
    $200: có 3, cần ≤ 0 -> lấy 0, còn 100
    $100: có 0, cần ≤ 1 -> lấy 0, còn 100
    $ 50: có 1, cần ≤ 2 -> lấy 1, còn 50
    $ 20: có 0, cần ≤ 2 -> lấy 0, còn 50
    -> còn dư 50 => [-1], máy giữ nguyên
withdraw(600,) -> [-1] | máy: [0, 1, 0, 3, 1]
  withdraw(550) với máy [0, 1, 0, 3, 1]
    $500: có 1, cần ≤ 1 -> lấy 1, còn 50
    $200: có 3, cần ≤ 0 -> lấy 0, còn 50
    $100: có 0, cần ≤ 0 -> lấy 0, còn 50
    $ 50: có 1, cần ≤ 1 -> 

Trace thêm một kịch bản **tự dựng** (đã đối chiếu bằng `ATMBrute`): nạp `[3, 1, 0, 0, 0]`, rút `110` (= 50 + 3×20) rồi rút `60` (máy chỉ còn `0×20`… xem kết quả).

In [10]:
own_ops = ["ATM", "deposit", "withdraw", "withdraw", "deposit", "withdraw"]
own_args = [[], [[3, 1, 0, 0, 0]], [110], [60], [[3, 0, 0, 0, 0]], [60]]
atm = ATMTrace()
for op, arg in zip(own_ops[1:], own_args[1:]):
    res = getattr(atm, op)(*arg); print(f"{op}{tuple(arg)} -> {res}")
assert run(ATMGreedy, own_ops, own_args) == run(ATMBrute, own_ops, own_args)
print("khớp ATMBrute:", run(ATMBrute, own_ops, own_args))

deposit([3, 1, 0, 0, 0],) -> None
  withdraw(110) với máy [3, 1, 0, 0, 0]
    $500: có 0, cần ≤ 0 -> lấy 0, còn 110
    $200: có 0, cần ≤ 0 -> lấy 0, còn 110
    $100: có 0, cần ≤ 1 -> lấy 0, còn 110
    $ 50: có 1, cần ≤ 2 -> lấy 1, còn 60
    $ 20: có 3, cần ≤ 3 -> lấy 3, còn 0
    -> thành công [3, 1, 0, 0, 0] | máy còn [0, 0, 0, 0, 0]
withdraw(110,) -> [3, 1, 0, 0, 0]
  withdraw(60) với máy [0, 0, 0, 0, 0]
    $500: có 0, cần ≤ 0 -> lấy 0, còn 60
    $200: có 0, cần ≤ 0 -> lấy 0, còn 60
    $100: có 0, cần ≤ 0 -> lấy 0, còn 60
    $ 50: có 0, cần ≤ 1 -> lấy 0, còn 60
    $ 20: có 0, cần ≤ 3 -> lấy 0, còn 60
    -> còn dư 60 => [-1], máy giữ nguyên
withdraw(60,) -> [-1]
deposit([3, 0, 0, 0, 0],) -> None
  withdraw(60) với máy [3, 0, 0, 0, 0]
    $500: có 0, cần ≤ 0 -> lấy 0, còn 60
    $200: có 0, cần ≤ 0 -> lấy 0, còn 60
    $100: có 0, cần ≤ 0 -> lấy 0, còn 60
    $ 50: có 0, cần ≤ 1 -> lấy 0, còn 60
    $ 20: có 3, cần ≤ 3 -> lấy 3, còn 0
    -> thành công [3, 0, 0, 0, 0] | máy c

## 9. Code nộp LeetCode

In [11]:
class ATM:
    def __init__(self):
        self.d = [20, 50, 100, 200, 500]
        self.cnt = [0] * 5

    def deposit(self, banknotesCount: List[int]) -> None:
        for i, x in enumerate(banknotesCount):
            self.cnt[i] += x

    def withdraw(self, amount: int) -> List[int]:
        take = [0] * 5
        for i in range(4, -1, -1):                       # ưu tiên mệnh giá lớn
            take[i] = min(self.cnt[i], amount // self.d[i])
            amount -= take[i] * self.d[i]
        if amount:                                       # không trả đủ: không rút gì
            return [-1]
        for i in range(5):                               # commit
            self.cnt[i] -= take[i]
        return take

got = run(ATM, example_ops, example_args)
assert got == example_out
print("Ví dụ đề: OK", got)

Ví dụ đề: OK [None, None, [0, 0, 1, 0, 1], None, [-1], [0, 1, 0, 0, 1]]


## 10. Vì sao thuật toán đúng

**Tương đương với đặc tả.** Đặc tả (diễn đạt bằng `ATMBrute`): ở mệnh giá `d_i` (từ lớn tới nhỏ), lấy từng tờ chừng nào còn tờ và `d_i ≤ rem`. Vòng `while` này dừng sau đúng `t = min(cnt_i, ⌊rem/d_i⌋)` lần:
- Nếu `cnt_i ≤ ⌊rem/d_i⌋`: lấy hết `cnt_i` tờ (sau mỗi tờ `rem` vẫn `≥ d_i` cho tới tờ cuối) rồi dừng vì hết tờ.
- Ngược lại: sau `⌊rem/d_i⌋` tờ, `rem` còn `rem mod d_i < d_i` ⇒ dừng.

Do đó `take[i]` của Cách 2 bằng đúng số tờ đặc tả lấy, và `rem` sau mỗi mệnh giá giống hệt nhau ⇒ cùng kết quả cuối (thành công với cùng `take`, hoặc cùng thất bại).

**Nguyên tử.** Cách 2 không sửa `cnt` trong vòng tính `take`; chỉ trừ khi `amount == 0`. Nhánh `[-1]` thoát trước mọi thay đổi ⇒ trạng thái giữ nguyên, đúng yêu cầu "do not withdraw any banknotes".

**Không âm.** `take[i] ≤ cnt[i]` nên sau commit `cnt[i] ≥ 0` — bất biến được bảo toàn. Bộ test so Cách 2/3/4 với `ATMBrute` trên chuỗi lệnh ngẫu nhiên.

## 11. Phân tích Time / Space

| Cách | `deposit` | `withdraw` | Space |
|---|---|---|---|
| 1. Từng tờ | `O(5)` | `O(số tờ đưa ra)` ≤ `O(amount/20)` | `O(1)` |
| 2. Chia nguyên | `O(5)` | `O(5)` | `O(1)` |
| 3. + tổng tiền | `O(5)` | `O(5)` | `O(1)` |
| 4. Tổng quát | `O(D)` | `O(D)` | `O(D)` |

Đo: 5000 lệnh ngẫu nhiên với số tờ và số tiền lớn (bản từng tờ không chạy nổi kích thước này):

In [12]:
rng = random.Random(2241)
big_ops, big_args = ["ATM"], [[]]
for _ in range(5000):
    if rng.random() < 0.4:
        big_ops.append("deposit"); big_args.append([[rng.randint(0, 10**5) for _ in range(5)]])
    else:
        big_ops.append("withdraw"); big_args.append([rng.randint(1, 10**9) // 10 * 10])
for name, cls in [("greedy", ATMGreedy), ("with total", ATMWithTotal), ("general", ATMGeneral), ("ATM", ATM)]:
    t = time.perf_counter(); res = run(cls, big_ops, big_args)
    ok = sum(1 for r in res if isinstance(r, list) and r != [-1])
    print(f"{name:10s}: 5000 lệnh trong {time.perf_counter() - t:.4f}s, {ok} lần rút thành công")
assert run(ATMGreedy, big_ops, big_args) == run(ATM, big_ops, big_args) == run(ATMGeneral, big_ops, big_args) == run(ATMWithTotal, big_ops, big_args)

greedy    : 5000 lệnh trong 0.0033s, 462 lần rút thành công
with total: 5000 lệnh trong 0.0025s, 462 lần rút thành công
general   : 5000 lệnh trong 0.0036s, 462 lần rút thành công
ATM       : 5000 lệnh trong 0.0033s, 462 lần rút thành công


## 12. Bộ test

**Command runner + so sánh brute:** sinh chuỗi lệnh ngẫu nhiên (seed cố định) với số tờ nhỏ (để `ATMBrute` chạy nhanh), chạy song song mọi cài đặt và so **từng kết quả** cùng **trạng thái máy** sau mỗi lệnh. Thêm các kịch bản biên viết tay.

In [13]:
def gen_ops(rng, length, max_notes=4, max_amount=1500):
    ops, args = ["ATM"], [[]]
    for _ in range(length):
        if rng.random() < 0.45:
            ops.append("deposit"); args.append([[rng.randint(0, max_notes) for _ in range(5)]])
        else:
            ops.append("withdraw"); args.append([rng.choice([rng.randint(1, max_amount), rng.randint(1, max_amount // 10) * 10])])
    return ops, args

def state_of(obj):
    if isinstance(obj, ATMGeneral): return [obj.cnt[d] for d in obj.denoms]
    return list(obj.cnt)

impls = [ATMGreedy, ATMWithTotal, ATMGeneral, ATM]
rng = random.Random(42)
for _ in range(400):
    ops, args = gen_ops(rng, rng.randint(1, 25))
    objs = [ATMBrute()] + [cls() for cls in impls]
    for op, arg in zip(ops[1:], args[1:]):
        results = [getattr(o, op)(*arg) for o in objs]
        assert all(r == results[0] for r in results), (op, arg, results)
        states = [state_of(o) for o in objs]
        assert all(s == states[0] for s in states), (op, arg, states)
print("400 chuỗi lệnh ngẫu nhiên: mọi cài đặt khớp ATMBrute (kết quả + trạng thái)")

400 chuỗi lệnh ngẫu nhiên: mọi cài đặt khớp ATMBrute (kết quả + trạng thái)


In [14]:
edge_cases = [
    (["ATM", "deposit", "withdraw"], [[], [[0, 0, 0, 0, 0]], [20]], [None, None, [-1]]),
    (["ATM", "deposit", "withdraw"], [[], [[3, 0, 0, 0, 0]], [60]], [None, None, [3, 0, 0, 0, 0]]),
    (["ATM", "deposit", "withdraw"], [[], [[3, 1, 0, 0, 0]], [60]], [None, None, [-1]]),           # 50 trước rồi kẹt 10
    (["ATM", "deposit", "withdraw"], [[], [[0, 0, 0, 3, 1]], [600]], [None, None, [-1]]),          # ví dụ trong đề
    (["ATM", "deposit", "withdraw", "withdraw"], [[], [[0, 0, 0, 0, 2]], [500], [500]], [None, None, [0, 0, 0, 0, 1], [0, 0, 0, 0, 1]]),
    (["ATM", "deposit", "withdraw"], [[], [[10**9, 0, 0, 0, 0]], [2 * 10**9]], [None, None, [10**8, 0, 0, 0, 0]]),
    (["ATM", "deposit", "withdraw"], [[], [[1, 1, 1, 1, 1]], [870]], [None, None, [1, 1, 1, 1, 1]]),
]
for ops, args, exp in edge_cases:
    for cls in impls:
        assert run(cls, ops, args) == exp, (cls.__name__, ops, args, run(cls, ops, args))
    if max(sum(a[0]) for o, a in zip(ops, args) if o == "deposit") < 100:
        assert run(ATMBrute, ops, args) == exp
print("Kịch bản biên: OK")

Kịch bản biên: OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Trừ tờ ngay trong vòng lặp, thất bại thì không hoàn tác

In [15]:
class WrongNoRollback(ATMGreedy):
    def withdraw(self, amount):
        take = [0] * 5
        for i in range(4, -1, -1):
            take[i] = min(self.cnt[i], amount // DENOMS[i])
            self.cnt[i] -= take[i]                                # SAI: sửa trạng thái trước khi biết thành công
            amount -= take[i] * DENOMS[i]
        return [-1] if amount else take

print("sai:", run(WrongNoRollback, example_ops, example_args))
print("đúng:", example_out)

sai: [None, None, [0, 0, 1, 0, 1], None, [-1], [-1]]
đúng: [None, None, [0, 0, 1, 0, 1], None, [-1], [0, 1, 0, 0, 1]]


In [16]:
print("sửa (tính trước, commit sau):", run(ATMGreedy, example_ops, example_args))

sửa (tính trước, commit sau): [None, None, [0, 0, 1, 0, 1], None, [-1], [0, 1, 0, 0, 1]]


### Lỗi 2 — Dùng quy hoạch động "đổi tiền" thay vì luật ưu tiên tờ lớn

Bản sai tìm **bất kỳ** tổ hợp trả đủ (ưu tiên nhiều tờ lớn theo thứ tự từ điển). Nó chấp nhận `600 = 3×200` trong khi đề bắt từ chối.

In [17]:
class WrongAnyCombination(ATMGreedy):
    def withdraw(self, amount):
        def search(i, rem):                                       # thử số tờ mệnh giá i từ nhiều đến ít
            if rem == 0: return [0] * 5
            if i < 0: return None
            for t in range(min(self.cnt[i], rem // DENOMS[i]), -1, -1):
                sub = search(i - 1, rem - t * DENOMS[i])
                if sub is not None:
                    sub[i] = t; return sub
            return None
        take = search(4, amount)                                  # SAI: đề không cho chọn cách khác
        if take is None: return [-1]
        for i in range(5): self.cnt[i] -= take[i]
        return take

print("sai:", run(WrongAnyCombination, example_ops, example_args))
print("đúng:", example_out)

sai: [None, None, [0, 0, 1, 0, 1], None, [0, 0, 0, 3, 0], [0, 1, 0, 0, 1]]
đúng: [None, None, [0, 0, 1, 0, 1], None, [-1], [0, 1, 0, 0, 1]]


In [18]:
print("sửa (đúng luật ưu tiên lớn):", run(ATM, example_ops, example_args))

sửa (đúng luật ưu tiên lớn): [None, None, [0, 0, 1, 0, 1], None, [-1], [0, 1, 0, 0, 1]]


### Lỗi 3 — Duyệt mệnh giá từ **nhỏ** tới lớn

Nhầm thứ tự: mảng nạp theo thứ tự `$20 → $500`, nhưng khi rút phải đi **ngược** lại. Tìm tự động chuỗi lệnh ngắn nhất làm lộ lỗi.

In [19]:
class WrongAscending(ATMGreedy):
    def withdraw(self, amount):
        take = [0] * 5
        for i in range(5):                                        # SAI: phải range(4, -1, -1)
            take[i] = min(self.cnt[i], amount // DENOMS[i]); amount -= take[i] * DENOMS[i]
        if amount: return [-1]
        for i in range(5): self.cnt[i] -= take[i]
        return take

def find_counterexample(wrong_cls, seed=0, tries=3000):
    r = random.Random(seed)
    for length in range(1, 8):
        for _ in range(tries // 7):
            ops, args = gen_ops(r, length, max_notes=3, max_amount=700)
            if run(wrong_cls, ops, args) != run(ATMBrute, ops, args):
                return ops, args
    return None

ce = find_counterexample(WrongAscending)
print("phản ví dụ:", list(zip(ce[0], ce[1])))
print("sai :", run(WrongAscending, *ce)); print("đúng:", run(ATMBrute, *ce))

phản ví dụ: [('ATM', []), ('deposit', [[2, 3, 3, 2, 0]]), ('withdraw', [450])]
sai : [None, None, [-1]]
đúng: [None, None, [0, 1, 0, 2, 0]]


In [20]:
print("sửa:", run(ATM, *ce))

sửa: [None, None, [0, 1, 0, 2, 0]]


### Lỗi 4 — Trả về tham chiếu tới list nội bộ / lưu list của người gọi

Bản sai `deposit` lưu thẳng list người gọi (lần đầu) và `withdraw` trả về list mà sau đó còn tái sử dụng. Người gọi sửa list ⇒ trạng thái máy bị đổi theo.

In [21]:
class WrongAliasing:
    def __init__(self):
        self.cnt = None
    def deposit(self, banknotesCount):
        if self.cnt is None:
            self.cnt = banknotesCount                             # SAI: giữ tham chiếu list của người gọi
        else:
            for i in range(5): self.cnt[i] += banknotesCount[i]
    def withdraw(self, amount):
        return ATMGreedy.withdraw(self, amount)

notes = [0, 0, 0, 0, 2]
w = WrongAliasing(); w.deposit(notes)
notes[4] = 0                                                      # người gọi tái sử dụng list của họ
print("sai: rút 500 ->", w.withdraw(500), "(máy 'quên' 2 tờ $500 vì list bị sửa từ bên ngoài)")

sai: rút 500 -> [-1] (máy 'quên' 2 tờ $500 vì list bị sửa từ bên ngoài)


In [22]:
notes = [0, 0, 0, 0, 2]
a = ATM(); a.deposit(notes); notes[4] = 0
print("sửa (cộng từng phần tử vào list riêng):", a.withdraw(500))

sửa (cộng từng phần tử vào list riêng): [0, 0, 0, 0, 1]


### Lỗi 5 — Trả về `[]` hoặc `None` thay vì `[-1]` khi thất bại

In [23]:
class WrongFailValue(ATMGreedy):
    def withdraw(self, amount):
        res = ATMGreedy.withdraw(self, amount)
        return [] if res == [-1] else res                         # SAI: đề yêu cầu [-1]

print("sai:", run(WrongFailValue, example_ops, example_args)[4], "| đúng:", example_out[4])

sai: [] | đúng: [-1]


### Lỗi 6 — Từ chối sớm bằng tổng tiền nhưng quên cập nhật `total` khi rút

In [24]:
class WrongStaleTotal(ATMWithTotal):
    def withdraw(self, amount):
        if amount > self.total: return [-1]
        take = [0] * 5; rem = amount
        for i in range(4, -1, -1):
            take[i] = min(self.cnt[i], rem // DENOMS[i]); rem -= take[i] * DENOMS[i]
        if rem: return [-1]
        for i in range(5): self.cnt[i] -= take[i]
        return take                                               # SAI: quên self.total -= amount

class WrongStaleTotalStrict(WrongStaleTotal):
    """Biến thể còn tệ hơn: tin hoàn toàn vào total, chỉ kiểm total rồi tham lam không kiểm phần dư."""
    def withdraw(self, amount):
        if amount > self.total: return [-1]
        take = [0] * 5; rem = amount
        for i in range(4, -1, -1):
            take[i] = min(self.cnt[i], rem // DENOMS[i]); rem -= take[i] * DENOMS[i]
        for i in range(5): self.cnt[i] -= take[i]
        return take                                               # SAI kép

ce6 = find_counterexample(WrongStaleTotalStrict, seed=1)
print("phản ví dụ:", list(zip(ce6[0], ce6[1])))
print("sai :", run(WrongStaleTotalStrict, *ce6)); print("đúng:", run(ATMBrute, *ce6))
ce6b = find_counterexample(WrongStaleTotal, seed=2)
print("WrongStaleTotal (chỉ quên cập nhật total) có phản ví dụ?", ce6b is not None,
      "-> vô hại vì phần kiểm rem vẫn đúng; total chỉ là bộ lọc sớm (nhưng nó trở nên vô dụng)")

phản ví dụ: [('ATM', []), ('deposit', [[1, 3, 3, 3, 2]]), ('withdraw', [56])]
sai : [None, None, [0, 1, 0, 0, 0]]
đúng: [None, None, [-1]]
WrongStaleTotal (chỉ quên cập nhật total) có phản ví dụ? False -> vô hại vì phần kiểm rem vẫn đúng; total chỉ là bộ lọc sớm (nhưng nó trở nên vô dụng)


In [25]:
print("sửa:", run(ATMWithTotal, *ce6))

sửa: [None, None, [-1]]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài "Design" có thao tác **có thể thất bại** ⇒ luôn tính trên bản nháp rồi commit.
- Đọc kỹ ví dụ phản trực giác trong đề (ở đây: từ chối `600` dù `3×200` khả thi) — đó là gợi ý về luật nghiệp vụ.
- Số lượng tới `10^9` ⇒ không lặp từng đơn vị; dùng `min(cnt, rem // d)`.
- Hằng số mệnh giá đặt thành list `[20, 50, 100, 200, 500]` và duyệt `range(4, -1, -1)`.
- Kiểm thử lớp thiết kế: **command runner** + oracle brute + so trạng thái sau mỗi lệnh.

## 15. Biến thể của bài

1. **Cho phép chọn tổ hợp bất kỳ** (máy "thông minh"): bài đổi tiền có giới hạn số tờ ⇒ DP/backtracking (chính là `WrongAnyCombination` — đúng cho biến thể này).
2. **Tối thiểu số tờ** khi trả: DP bounded knapsack.
3. **Nhiều máy / nhiều người dùng đồng thời:** cần khóa (lock) để đảm bảo nguyên tử.
4. **Lịch sử giao dịch / hoàn tác:** lưu nhật ký các `take` để `undo`.
5. **LeetCode 860 — Lemonade Change:** tham lam trả tiền thối với mệnh giá 5/10/20.

Minh họa biến thể 1 + 2 — máy "thông minh" trả bằng **ít tờ nhất** (tự dựng, kiểm bằng vét cạn):

In [26]:
def min_notes_plan(cnt, amount):
    best = None
    def rec(i, rem, cur):
        nonlocal best
        if rem == 0:
            if best is None or sum(cur) < sum(best): best = cur[:]
            return
        if i < 0: return
        for t in range(min(cnt[i], rem // DENOMS[i]), -1, -1):
            cur[i] = t; rec(i - 1, rem - t * DENOMS[i], cur)
        cur[i] = 0
    rec(4, amount, [0] * 5)
    return best

def min_notes_brute(cnt, amount):
    best = None
    for t in product(*[range(c + 1) for c in cnt]):
        if sum(x * d for x, d in zip(t, DENOMS)) == amount and (best is None or sum(t) < sum(best)):
            best = list(t)
    return best

rngv = random.Random(15)
for _ in range(150):
    cnt = [rngv.randint(0, 3) for _ in range(5)]; amt = rngv.randint(1, 120) * 10
    a, b = min_notes_plan(cnt, amt), min_notes_brute(cnt, amt)
    assert (a is None) == (b is None) and (a is None or sum(a) == sum(b))
print("máy thông minh với [0,1,0,3,1], 600 ->", min_notes_plan([0, 1, 0, 3, 1], 600), "| máy theo đề -> [-1]")

máy thông minh với [0,1,0,3,1], 600 -> [0, 0, 0, 3, 0] | máy theo đề -> [-1]


## 16. So sánh các cách

| Cách | Ý tưởng | `withdraw` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Từng tờ | mô phỏng sát đặc tả | `O(amount/20)` | `O(1)` | oracle kiểm thử |
| 2. Chia nguyên | `min(cnt, rem // d)` + commit | `O(1)` | `O(1)` | **nộp bài** |
| 3. + tổng tiền | từ chối sớm | `O(1)` | `O(1)` | minh họa bất biến phụ |
| 4. Tổng quát | hàm thuần + dict | `O(D)` | `O(D)` | mệnh giá thay đổi |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `ATM(); deposit([5, 0, 0, 0, 0]); withdraw(70)`.
2. **Tính tay:** máy `[2, 2, 2, 2, 2]`, rút `730` — kết quả và trạng thái sau.
3. **Điền TODO:** `can_withdraw(cnt, amount)` — hàm thuần trả `True/False` theo luật của đề, không sửa `cnt`.
4. **Tự viết lại:** viết lại lớp `ATM` không nhìn code.
5. **Mini challenge:** thêm `undo()` vào ATM — hoàn tác lệnh `deposit`/`withdraw` thành công gần nhất (rút thất bại không tính). Kiểm bằng chuỗi lệnh ngẫu nhiên so với bản lưu toàn bộ lịch sử trạng thái.

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def can_withdraw(cnt, amount):
    # TODO
    pass

In [28]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
class MyATM:
    def __init__(self):
        # TODO
        pass
    def deposit(self, banknotesCount):
        # TODO
        pass
    def withdraw(self, amount):
        # TODO
        pass

In [29]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
class ATMUndo:
    def __init__(self):
        # TODO
        pass
    def undo(self):
        # TODO
        pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Bắt đầu từ `$500`… đều 0 tờ. (2) `$50`: 0 tờ; `$20`: `min(5, 70//20 = 3) = 3` ⇒ còn 10. (3) `[-1]`.

**Bài 2.** (1) `$500`: 1 tờ, còn 230. (2) `$200`: 1, còn 30; `$100`: 0; `$50`: 0; `$20`: 1, còn 10. (3) `[-1]`, máy giữ nguyên `[2,2,2,2,2]`.

**Bài 3.** (1) Vòng `range(4, -1, -1)`. (2) `rem -= min(cnt[i], rem // d) * d`. (3) Trả `rem == 0`.

**Bài 4.** (1) `cnt = [0]*5`. (2) `deposit` cộng từng phần tử. (3) `withdraw` tính `take`, commit khi thành công.

**Bài 5.** (1) Mỗi thao tác thành công lưu `(loại, vector)` vào stack. (2) `undo` của deposit là trừ vector; của withdraw là cộng lại. (3) Oracle: lưu bản sao trạng thái trước mỗi thao tác thành công, `undo` = pop.

## 19. Đáp án tham khảo

In [30]:
# Bài 1, 2
assert run(ATM, ["ATM", "deposit", "withdraw"], [[], [[5, 0, 0, 0, 0]], [70]]) == [None, None, [-1]]
a = ATMTrace(); a.deposit([2, 2, 2, 2, 2]); r = a.withdraw(730)
assert r == [-1] and a.cnt == [2, 2, 2, 2, 2]
print("Bài 1: [-1]; Bài 2:", r, a.cnt)

  withdraw(730) với máy [2, 2, 2, 2, 2]
    $500: có 2, cần ≤ 1 -> lấy 1, còn 230
    $200: có 2, cần ≤ 1 -> lấy 1, còn 30
    $100: có 2, cần ≤ 0 -> lấy 0, còn 30
    $ 50: có 2, cần ≤ 0 -> lấy 0, còn 30
    $ 20: có 2, cần ≤ 1 -> lấy 1, còn 10
    -> còn dư 10 => [-1], máy giữ nguyên
Bài 1: [-1]; Bài 2: [-1] [2, 2, 2, 2, 2]


In [31]:
# Bài 3, 4
def can_withdraw_ans(cnt, amount):
    for i in range(4, -1, -1):
        amount -= min(cnt[i], amount // DENOMS[i]) * DENOMS[i]
    return amount == 0

class MyATMAns:
    def __init__(self): self.c = [0] * 5
    def deposit(self, banknotesCount):
        for i in range(5): self.c[i] += banknotesCount[i]
    def withdraw(self, amount):
        t = [0] * 5
        for i in range(4, -1, -1):
            t[i] = min(self.c[i], amount // DENOMS[i]); amount -= t[i] * DENOMS[i]
        if amount: return [-1]
        for i in range(5): self.c[i] -= t[i]
        return t

rng3 = random.Random(33)
for _ in range(300):
    cnt = [rng3.randint(0, 4) for _ in range(5)]; amt = rng3.randint(1, 2000); before = cnt[:]
    b = ATMBrute(); b.deposit(cnt)
    assert can_withdraw_ans(cnt, amt) == (b.withdraw(amt) != [-1]) and cnt == before
    ops, args = gen_ops(rng3, rng3.randint(1, 15))
    assert run(MyATMAns, ops, args) == run(ATMBrute, ops, args)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [32]:
# Bài 5
class ATMUndoAns(ATM):
    def __init__(self):
        super().__init__(); self.log = []
    def deposit(self, banknotesCount):
        super().deposit(banknotesCount); self.log.append(list(banknotesCount))
    def withdraw(self, amount):
        res = super().withdraw(amount)
        if res != [-1]: self.log.append([-x for x in res])
        return res
    def undo(self):
        if not self.log: return False
        delta = self.log.pop()
        for i in range(5): self.cnt[i] -= delta[i]
        return True

class ATMUndoOracle(ATMBrute):
    def __init__(self):
        super().__init__(); self.hist = []
    def deposit(self, banknotesCount):
        self.hist.append(self.cnt[:]); super().deposit(banknotesCount)
    def withdraw(self, amount):
        snap = self.cnt[:]; res = super().withdraw(amount)
        if res != [-1]: self.hist.append(snap)
        return res
    def undo(self):
        if not self.hist: return False
        self.cnt = self.hist.pop(); return True

rng5 = random.Random(55)
for _ in range(300):
    a, o = ATMUndoAns(), ATMUndoOracle()
    for _ in range(rng5.randint(1, 20)):
        x = rng5.random()
        if x < 0.35: arg = [rng5.randint(0, 3) for _ in range(5)]; a.deposit(arg); o.deposit(arg)
        elif x < 0.75: amt = rng5.randint(1, 1200); assert a.withdraw(amt) == o.withdraw(amt)
        else: assert a.undo() == o.undo()
        assert a.cnt == o.cnt
print("Bài 5: OK — undo khớp oracle lưu lịch sử trạng thái")

Bài 5: OK — undo khớp oracle lưu lịch sử trạng thái


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tham lam theo luật + commit":**

```python
take = [0] * 5
for i in range(4, -1, -1):                 # lớn -> nhỏ
    take[i] = min(cnt[i], amount // d[i])
    amount -= take[i] * d[i]
if amount: return [-1]                     # chưa sửa gì
for i in range(5): cnt[i] -= take[i]       # commit
return take
```

In [33]:
def template_withdraw(cnt, amount, d=DENOMS):
    take = [0] * 5
    for i in range(4, -1, -1):
        take[i] = min(cnt[i], amount // d[i]); amount -= take[i] * d[i]
    if amount: return [-1]
    for i in range(5): cnt[i] -= take[i]
    return take
c = [0, 1, 0, 3, 1]
print(template_withdraw(c, 600), c, template_withdraw(c, 550), c)

[-1] [0, 0, 0, 3, 0] [0, 1, 0, 0, 1] [0, 0, 0, 3, 0]


**Checklist:**

- [ ] Duyệt mệnh giá từ **lớn** tới nhỏ; mảng vào/ra theo thứ tự `$20 … $500`.
- [ ] `take = min(cnt, rem // d)` — không lặp từng tờ.
- [ ] Thất bại ⇒ `[-1]` và **không** sửa trạng thái.
- [ ] Không thay luật bằng DP "đổi tiền".
- [ ] Không giữ tham chiếu list của người gọi; trả list mới.
- [ ] Kiểm bằng command runner + oracle trên chuỗi lệnh ngẫu nhiên.